# 04 — ToN-IoT Model Training

**Goal:** Train five classifiers using the full frozen training dataset.

**Input:** Core10 train and validation CSV.GZ files from Notebook 03.

**Output:** Saved models, validation metrics, resource measurements and comparison graphs.

**Rule:** The held-out test data stays untouched. Excel copies are for reading only.

In [ ]:
from pathlib import Path
from time import perf_counter
from datetime import datetime, timezone
from threading import Thread, Event
import hashlib, inspect, json, os, gc, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, psutil, sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score, f1_score)

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
SPLITS = ROOT / 'data/splits/ton_iot'
TRAIN_FILE = SPLITS / 'ton_iot_core10_train_v3.csv.gz'
VALID_FILE = SPLITS / 'ton_iot_core10_validation_v3.csv.gz'
MANIFEST = SPLITS / 'ton_iot_core10_signature_split_manifest_v3.csv'
SPLIT_REPORT = ROOT / 'reports/audits/ton_iot_v3_split_report.json'
MODEL_DIR = ROOT / 'models/ton_iot_v3'
AUDITS = ROOT / 'reports/audits'
METRICS = ROOT / 'reports/metrics'
FIGURES = ROOT / 'reports/figures'
SEED = 42
MODEL_NAMES = ['decision_tree', 'random_forest', 'extra_trees', 'lightgbm', 'adaboost']
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing', 'unknown', '<missing>'}
SINGLE_FLOW_RUNS = 100  # Samples only latency; all rows are trained/validated.

for path in [TRAIN_FILE, VALID_FILE, SPLIT_REPORT, MANIFEST]:
    if not path.is_file():
        raise FileNotFoundError(f'Missing Notebook 03 output: {path}')
report = json.loads(SPLIT_REPORT.read_text(encoding='utf-8'))
if report.get('dataset') != 'ToN-IoT':
    raise ValueError('This is not a ToN-IoT split report.')
counts = {p: int(report['counts'][p]['rows']) for p in ['train', 'validation', 'test']}
if sum(counts.values()) != 211043 or report.get('total_rows_preserved') != 211043:
    raise ValueError(f'Expected all 211,043 records; got {counts}. Do not resplit silently.')
if report.get('normalized_fingerprint_overlap_groups') != 0:
    raise ValueError('Core10 feature groups overlap between partitions.')
if report.get('test_scored', False):
    raise ValueError('Held-out test set already scored; investigate before continuing.')
sha = hashlib.sha256()
with MANIFEST.open('rb') as fh:
    for block in iter(lambda: fh.read(8 * 1024 * 1024), b''):
        sha.update(block)
if sha.hexdigest() != report.get('manifest_sha256'):
    raise ValueError('Frozen manifest changed after Notebook 03. Stop and investigate.')
FEATURES = list(report['model_features'])
CAT = list(report['categorical_features'])
NUM = [f for f in FEATURES if f not in CAT]
if len(FEATURES) != 10 or len(CAT) != 3 or len(NUM) != 7:
    raise ValueError(f'Unexpected Core10 schema: {len(FEATURES)} features, {len(CAT)} categorical.')
if len(set(FEATURES)) != 10 or not set(CAT).issubset(FEATURES):
    raise ValueError('Duplicate features or unexpected categorical list.')
for d in (MODEL_DIR, AUDITS, METRICS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
print('ToN-IoT | Features:',len(FEATURES),'| All dataset rows:',sum(counts.values()))
print('Training:',counts['train'],'| Validation:',counts['validation'])
print('TEST file is not read. Excel files are for manual review.')

## 1. Load training and validation

Read every record from both partitions. Keep labels and row IDs out of model inputs.

In [ ]:
FIELDS = ['csv_row_index_0based'] + FEATURES + ['label', 'type']
train = pd.read_csv(TRAIN_FILE, dtype=str, keep_default_na=False, low_memory=False)
valid = pd.read_csv(VALID_FILE, dtype=str, keep_default_na=False, low_memory=False)
for part, frame in [('train', train), ('validation', valid)]:
    if frame.columns.tolist() != FIELDS:
        raise ValueError(f'{part}: incorrect column names/order; expected {FIELDS}')
    if len(frame) != counts[part]:
        raise ValueError(f'{part}: {len(frame):,} rows; expected {counts[part]:,}')
    if frame['csv_row_index_0based'].duplicated().any():
        raise ValueError(f'{part}: duplicate original row IDs.')
    if not frame['label'].isin(['0','1']).all():
        raise ValueError(f'{part}: binary label must be 0 or 1.')
    target = report['counts'][part]
    if (int((frame['label']=='0').sum()) != int(target['normal']) or
        int((frame['label']=='1').sum()) != int(target['attack'])):
        raise ValueError(f'{part}: actual label counts disagree with frozen report.')
if set(train['csv_row_index_0based']).intersection(valid['csv_row_index_0based']):
    raise ValueError('Train and validation original row IDs overlap.')

y_train = train['label'].astype(np.uint8).to_numpy()
y_valid = valid['label'].astype(np.uint8).to_numpy()
X_train_raw = train[FEATURES].copy()
X_valid_raw = valid[FEATURES].copy()
del train, valid
gc.collect()
print(f'Full training: {len(y_train):,} | Full validation: {len(y_valid):,}')
print('No rows sampled and no test data loaded.')

## 2. Prepare model inputs

Handle missing values and encode categories. The preprocessing is fitted using **training data only**.

In [ ]:
def clean_features(frame):
    out = frame[FEATURES].copy()
    for col in NUM:
        values = out[col].astype(str).str.strip()
        values = values.mask(values.str.casefold().isin(MISSING))
        out[col] = pd.to_numeric(values, errors='coerce').replace(
            [np.inf, -np.inf], np.nan).astype(np.float32)
    for col in CAT:
        values = out[col].astype(str).str.strip()
        out[col] = values.mask(values.str.casefold().isin(MISSING), '__MISSING__')
    return out

X_train = clean_features(X_train_raw)
X_valid = clean_features(X_valid_raw)
del X_train_raw, X_valid_raw
gc.collect()
ONE_FLOW = X_valid.iloc[[0]]

def build_preprocessor():
    ohe_kwargs = {'handle_unknown': 'ignore'}
    if 'sparse_output' in inspect.signature(OneHotEncoder).parameters:
        ohe_kwargs['sparse_output'] = True
    else:
        ohe_kwargs['sparse'] = True
    imputer_kwargs = {'strategy': 'median'}
    if 'keep_empty_features' in inspect.signature(SimpleImputer).parameters:
        imputer_kwargs['keep_empty_features'] = True
    return ColumnTransformer([
        ('numeric', SimpleImputer(**imputer_kwargs), NUM),
        ('category', OneHotEncoder(**ohe_kwargs), CAT)
    ], sparse_threshold=1.0)

process = psutil.Process(os.getpid())
def memory_mib():
    return process.memory_info().rss / (1024**2)
def cpu_seconds():
    t = process.cpu_times()
    return t.user + t.system

def measure_resources(fn):
    before = memory_mib()
    peak = [before]
    stop = Event()
    def monitor():
        while not stop.wait(0.02):
            peak[0] = max(peak[0], memory_mib())
    thread = Thread(target=monitor, daemon=True)
    c0 = cpu_seconds()
    t0 = perf_counter()
    thread.start()
    try:
        result = fn()
    finally:
        seconds = perf_counter() - t0
        cpu = max(0.0, cpu_seconds() - c0)
        stop.set()
        thread.join()
        peak[0] = max(peak[0], memory_mib())
    return result, {
        'seconds': round(seconds, 4),
        'process_cpu_pct': round(100 * cpu / max(seconds, 1e-9), 2),
        'peak_rss_mib': round(peak[0], 2),
        'rss_increase_mib': round(max(0, peak[0]-before), 2)
    }

print('Prepared training matrix:', X_train.shape, '| Validation:', X_valid.shape)
print('Process RAM includes loaded Python data/libraries; it is not model-only RAM.')

## 3. Shared evaluation function

Each model is trained on all training rows and checked on all validation rows. Save its results immediately.

In [ ]:
def evaluate_model(name, estimator):
    print('\n===', name.replace('_', ' ').title(), '===')
    pipeline = Pipeline([
        ('preprocessing', build_preprocessor()),
        ('classifier', estimator)
    ])
    _, training = measure_resources(lambda: pipeline.fit(X_train, y_train))
    predicted, predicting = measure_resources(lambda: pipeline.predict(X_valid))
    tn, fp, fn, tp = (int(x) for x in confusion_matrix(y_valid, predicted, labels=[0,1]).ravel())

    # Prediction timing includes preprocessing and model inference for one row.
    for _ in range(5):
        pipeline.predict(ONE_FLOW)
    sample_ids = np.linspace(0, len(X_valid)-1,
                             num=min(SINGLE_FLOW_RUNS, len(X_valid)), dtype=int)
    times_ms = []
    for index in sample_ids:
        row = X_valid.iloc[[int(index)]]
        start = perf_counter()
        pipeline.predict(row)
        times_ms.append((perf_counter()-start)*1000)

    model_path = MODEL_DIR / f'{name}_core10_v3.joblib'
    joblib.dump(pipeline, model_path, compress=3)
    metrics = {
        'model': name,
        'training_rows': int(len(y_train)),
        'validation_rows': int(len(y_valid)),
        'accuracy': round(float(accuracy_score(y_valid, predicted)), 6),
        'balanced_accuracy': round(float(balanced_accuracy_score(y_valid, predicted)), 6),
        'precision_attack': round(float(precision_score(y_valid, predicted, zero_division=0)), 6),
        'recall_attack': round(float(recall_score(y_valid, predicted, zero_division=0)), 6),
        'f1_attack': round(float(f1_score(y_valid, predicted, zero_division=0)), 6),
        'false_positive_rate': round(fp/max(tn+fp,1), 6),
        'false_negative_rate': round(fn/max(fn+tp,1), 6),
        'tn':tn, 'fp':fp, 'fn':fn, 'tp':tp
    }
    resources = {
        'model':name,
        'training_seconds':training['seconds'],
        'train_process_cpu_percent':training['process_cpu_pct'],
        'train_peak_process_ram_mib':training['peak_rss_mib'],
        'train_peak_ram_increase_mib':training['rss_increase_mib'],
        'validation_predict_seconds':predicting['seconds'],
        'predict_process_cpu_percent':predicting['process_cpu_pct'],
        'predict_peak_process_ram_mib':predicting['peak_rss_mib'],
        'batch_throughput_flows_per_second':round(len(y_valid)/max(predicting['seconds'], 1e-9), 1),
        'single_flow_p50_ms':round(float(np.percentile(times_ms,50)), 3),
        'single_flow_p95_ms':round(float(np.percentile(times_ms,95)), 3),
        'single_flow_runs':int(len(times_ms)),
        'model_size_mib':round(model_path.stat().st_size/(1024**2), 4)
    }
    top_features = []
    classifier = pipeline.named_steps['classifier']
    if hasattr(classifier, 'feature_importances_'):
        weights = classifier.feature_importances_
        names = pipeline.named_steps['preprocessing'].get_feature_names_out()
        top_features = [{'feature':str(names[i]),'importance':round(float(weights[i]),6)}
                        for i in np.argsort(weights)[::-1][:10]]

    artifact = {
        'experiment':'ToN-IoT Core10 V3 | validation only',
        'model':name,
        'created_utc':datetime.now(timezone.utc).isoformat(),
        'features':FEATURES,
        'target':'label',
        'label_mapping':{'0':'Normal','1':'Attack'},
        'seed':SEED,
        'frozen_manifest_sha256':report['manifest_sha256'],
        'model_file':model_path.name,
        'metrics':metrics,
        'resources':resources,
        'top_transformed_feature_importance':top_features,
        'versions':{'python':platform.python_version(),'scikit_learn':sklearn.__version__},
        'test_scored':False,
        'limitations':[
            'Results are validation-only, not final test accuracy.',
            'Service has many placeholders; possible label shortcut needs evaluation.',
            'Group-disjoint fingerprints do not prove session/capture independence.',
            'CPU and RAM measurements are from a laptop process, not Raspberry Pi.',
            'Feature importance is model-specific and not causal evidence.'
        ]
    }
    out = AUDITS / f'ton_iot_v3_{name}_training.json'
    out.write_text(json.dumps(artifact, indent=2), encoding='utf-8')

    fig, ax = plt.subplots(figsize=(5.5, 4.4))
    ConfusionMatrixDisplay.from_predictions(y_valid, predicted, labels=[0,1],
        display_labels=['Normal', 'Attack'], cmap='Blues', values_format='d', ax=ax)
    ax.set_title(f'{name.replace("_"," ").title()} — ToN-IoT validation')
    fig.tight_layout()
    fig.savefig(FIGURES / f'ton_iot_v3_{name}_confusion_matrix.png',
                dpi=170, bbox_inches='tight')
    plt.show()
    plt.close(fig)

    display(pd.DataFrame([metrics]).T.rename(columns={0:name}))
    display(pd.DataFrame([resources]).T.rename(columns={0:name}))
    if top_features:
        display(pd.DataFrame(top_features))
    print('Saved:', model_path.name, '|', out.name)
    return artifact

## 4. Decision Tree

Small single-tree baseline.

In [ ]:
decision_tree_result = evaluate_model('decision_tree',
    DecisionTreeClassifier(max_depth=14, min_samples_leaf=30,
                           class_weight='balanced', random_state=SEED))

## 5. Random Forest

Tree ensemble. Compare detection and resource cost.

In [ ]:
random_forest_result = evaluate_model('random_forest',
    RandomForestClassifier(n_estimators=100, max_depth=16, min_samples_leaf=20,
                           class_weight='balanced_subsample', random_state=SEED, n_jobs=-1))

## 6. Extra Trees

Randomized tree ensemble.

In [ ]:
extra_trees_result = evaluate_model('extra_trees',
    ExtraTreesClassifier(n_estimators=100, max_depth=16, min_samples_leaf=20,
                         class_weight='balanced', random_state=SEED, n_jobs=-1))

## 7. LightGBM

Boosted trees. Multicore CPU use can exceed 100%.

In [ ]:
from lightgbm import LGBMClassifier
lightgbm_result = evaluate_model('lightgbm',
    LGBMClassifier(n_estimators=160, num_leaves=31, learning_rate=0.05,
                   min_child_samples=30, random_state=SEED, n_jobs=4, verbosity=-1))

## 8. AdaBoost

Boost shallow decision trees.

In [ ]:
small_tree = DecisionTreeClassifier(max_depth=2, min_samples_leaf=30, random_state=SEED)
ada_args = {'n_estimators':100, 'learning_rate':0.5, 'random_state':SEED}
if 'estimator' in inspect.signature(AdaBoostClassifier).parameters:
    ada_args['estimator'] = small_tree
else:
    ada_args['base_estimator'] = small_tree
adaboost_result = evaluate_model('adaboost', AdaBoostClassifier(**ada_args))

## 9. Compare all models

Read saved results. This step does not retrain any model.

In [ ]:
results = []
for name in MODEL_NAMES:
    path = AUDITS / f'ton_iot_v3_{name}_training.json'
    if not path.is_file():
        raise FileNotFoundError(f'Model training report missing: {path.name}. Run its model cell.')
    record = json.loads(path.read_text(encoding='utf-8'))
    if record.get('frozen_manifest_sha256') != report['manifest_sha256']:
        raise ValueError(f'{name}: saved model report belongs to a different frozen split.')
    if (record['metrics']['training_rows'] != counts['train'] or
        record['metrics']['validation_rows'] != counts['validation']):
        raise ValueError(f'{name}: saved model was not trained on all current records.')
    results.append(record)

comparison = pd.DataFrame([r['metrics'] for r in results])
resources = pd.DataFrame([r['resources'] for r in results])
comparison.to_csv(METRICS / 'ton_iot_v3_five_model_validation.csv', index=False)
resources.to_csv(METRICS / 'ton_iot_v3_five_model_resources.csv', index=False)

fig, axes = plt.subplots(1, 2, figsize=(12,4.6))
labels = comparison['model'].str.replace('_',' ').str.title()
axes[0].bar(labels, comparison['f1_attack']*100)
axes[0].set_title('ToN-IoT — Attack F1 (validation)')
axes[0].set_ylabel('F1-score (%)')
axes[0].set_ylim(0, 105)
axes[1].bar(labels, comparison['false_positive_rate']*100)
axes[1].set_title('ToN-IoT — False positive rate (validation)')
axes[1].set_ylabel('False-positive rate (%)')
for a in axes:
    a.tick_params(axis='x', labelrotation=17)
    a.grid(axis='y', alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES / 'ton_iot_v3_model_detection.png', dpi=180, bbox_inches='tight')
plt.show(); plt.close(fig)

fig, axes = plt.subplots(2,2,figsize=(11,8))
for ax, (field,title,unit) in zip(axes.flat,[
    ('training_seconds','Training time','seconds'),
    ('train_peak_process_ram_mib','Peak training process RAM','MiB'),
    ('single_flow_p95_ms','Single-flow inference p95','ms'),
    ('model_size_mib','Model storage','MiB')]):
    ax.bar(resources['model'].str.replace('_',' ').str.title(), resources[field])
    ax.set_title(title);ax.set_ylabel(unit)
    ax.tick_params(axis='x',labelrotation=17)
    ax.grid(axis='y',alpha=.2)
fig.tight_layout()
fig.savefig(FIGURES / 'ton_iot_v3_model_resources.png', dpi=180, bbox_inches='tight')
plt.show(); plt.close(fig)

display(comparison[['model','accuracy','precision_attack','recall_attack',
    'f1_attack','false_positive_rate','false_negative_rate','tn','fp','fn','tp']])
display(resources[['model','training_seconds','train_process_cpu_percent',
    'train_peak_process_ram_mib','validation_predict_seconds',
    'batch_throughput_flows_per_second','single_flow_p95_ms','model_size_mib']])
print('Saved: five model reports, five confusion matrices, two metrics CSVs and two graphs.')
print('Test split was never read.')